# 🧹 02 — Data Cleaning & Feature Engineering

**Ziel:** Roh-Jobdaten in eine analyse-fähige Form bringen.

## 🤔 Warum Cleaning?

Die rohen API-Daten haben mehrere Probleme:
- **Stadt-Feld unzuverlässig** — manchmal leer, manchmal `"Köln, Deutschland"`, manchmal nur in der Beschreibung
- **Salary-Daten unstrukturiert** — als Freitext im Beschreibungstext (`"45.000 € - 60.000 € pro Jahr"`)
- **Skill-Information versteckt** — keine eigene Spalte, sondern als Lauftext in der Description
- **Job-Titel inkonsistent** — `"Data Analyst"`, `"Datenanalyst"`, `"Senior Data Analyst (m/w/d)"`, `"BI Analyst / Data Analyst"`

## 🎯 Plan
1. Roh-Daten laden
2. Schema-Migration für Legacy-Daten
3. Stadt aus 3 Quellen extrahieren
4. Junior-Erkennung über Keywords
5. Remote-Erkennung über Regex
6. Rollen-Klassifikation in Cluster
7. Vorher-Nachher-Vergleich

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

## 1️⃣ Roh-Daten laden

Wir laden die Master-Datei, die alle Sammel-Läufe akkumuliert hat. Das macht das Sammeln **inkrementell** — bestehende Jobs werden nicht neu gezogen.

In [ ]:
raw_path = ROOT / "data" / "raw" / "jobs_raw_master.csv"

if not raw_path.exists():
    print(f"❌ Datei fehlt: {raw_path}")
    print("   Erst Notebook 01 ausführen oder: python -m src.collect_jobs")
else:
    df_raw = pd.read_csv(raw_path)
    print(f"✅ Geladen: {len(df_raw)} Jobs")
    print(f"   Spalten: {list(df_raw.columns)[:10]}...")
    df_raw.head(3)

In [ ]:
# Diagnose: Wo sind Lücken?
missing = df_raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(1)
diagnosis = pd.DataFrame({"missing": missing, "missing_%": missing_pct})
diagnosis = diagnosis[diagnosis["missing"] > 0]
print("Fehlende Werte pro Spalte:")
diagnosis.head(15)

## 2️⃣ Stadt-Extraktion

Die Stadt-Information steckt **in mehreren Feldern unterschiedlicher Quellen**:
- `job_city` — wenn die API es liefert (Adzuna ja, Arbeitsagentur teilweise)
- `job_location` — kombiniertes Feld `"Köln, Nordrhein-Westfalen"`
- **Description** — letzte Rettung: Regex-Suche nach deutschen Großstädten

Das ist ein klassisches Cleaning-Problem: **eine Information, mehrere Fallback-Quellen, gewichtete Kombination**.

In [ ]:
import re

# Liste deutscher Großstädte für Regex-Fallback
GERMAN_CITIES = [
    "Berlin", "Hamburg", "München", "Köln", "Frankfurt", "Stuttgart",
    "Düsseldorf", "Leipzig", "Dortmund", "Essen", "Bremen", "Dresden",
    "Hannover", "Nürnberg", "Bonn", "Mannheim", "Karlsruhe", "Wiesbaden",
    "Münster", "Aachen", "Bielefeld", "Mainz", "Augsburg", "Wuppertal",
    "Leverkusen", "Heidelberg", "Freiburg",
]
CITY_PATTERN = re.compile(r"\b(" + "|".join(GERMAN_CITIES) + r")\b", re.IGNORECASE)


def extract_city(row) -> str:
    """Extrahiert die Stadt aus 3 möglichen Quellen."""
    # 1. Bevorzugt: explizites Feld
    city = str(row.get("job_city", "") or "").strip()
    if city and city.lower() not in ("nan", "", "none"):
        return city

    # 2. Aus job_location ("Köln, Nordrhein-Westfalen")
    loc = str(row.get("job_location", "") or "")
    if loc and "," in loc:
        return loc.split(",")[0].strip()

    # 3. Fallback: erste deutsche Stadt in der Description finden
    desc = str(row.get("job_description", "") or "")[:500]
    match = CITY_PATTERN.search(desc)
    if match:
        return match.group(0).capitalize()

    return "Unbekannt"


df_clean = df_raw.copy()
df_clean["job_city"] = df_clean.apply(extract_city, axis=1)

# Ergebnis prüfen
city_counts = df_clean["job_city"].value_counts().head(10)
print(f"Stadt erkannt für: {(df_clean['job_city'] != 'Unbekannt').sum()} / {len(df_clean)} Jobs\n")
print("Top 10 Städte:")
print(city_counts.to_string())

## 3️⃣ Junior-Erkennung

Klassisches Keyword-Matching mit deutschen und englischen Begriffen:

In [ ]:
JUNIOR_KEYWORDS = [
    "junior", "entry level", "graduate", "trainee",
    "berufseinstieg", "absolvent", "einsteiger", "werkstudent",
]

def is_junior(text: str) -> int:
    text = str(text).lower()
    return int(any(kw in text for kw in JUNIOR_KEYWORDS))

df_clean["is_junior"] = (
    df_clean["job_title"].fillna("")
    + " "
    + df_clean.get("job_description", pd.Series(dtype=str)).fillna("")
).apply(is_junior)

junior_count = df_clean["is_junior"].sum()
print(f"Junior-Stellen: {junior_count} ({junior_count/len(df_clean)*100:.1f}%)")

# Beispiele anzeigen
print("\n5 Beispiele (als Junior erkannt):")
df_clean[df_clean["is_junior"] == 1][["job_title", "employer_name"]].head(5)

## 4️⃣ Remote-Erkennung mit Regex

Hier wird's interessant. **"Remote möglich"** kann auf vielen Wegen ausgedrückt werden:
- Englisch: `remote`, `home office`, `wfh`, `mobile work`
- Deutsch: `mobiles Arbeiten`, `Homeoffice`, `ortsunabhängig`, `Telearbeit`

Wir nutzen **kompilierte Regex mit Wortgrenzen** — verhindert False Positives wie `"home"` in `"homepage"`.

In [ ]:
import re

REMOTE_PATTERNS = [
    r"\bremote\b",
    r"\bhome[\s\-]?office\b",
    r"\bhomeoffice\b",
    r"\bwork\s+from\s+home\b",
    r"\bwfh\b",
    r"\bmobile\s+work\b",
    r"\bmobiles?\s+arbeit(?:en)?\b",
    r"\bortsunabh[äa]ngig\b",
    r"\btele[\s\-]?arbeit\b",
    r"\bhybrid\b",
]

REMOTE_REGEX = re.compile("|".join(REMOTE_PATTERNS), flags=re.IGNORECASE)


def detect_remote(text: str) -> bool:
    return bool(REMOTE_REGEX.search(str(text)))


combined = df_clean["job_title"].fillna("") + " " + df_clean["job_description"].fillna("")
df_clean["is_remote_friendly"] = combined.apply(detect_remote)

remote_count = df_clean["is_remote_friendly"].sum()
print(f"Remote-fähige Stellen: {remote_count} ({remote_count/len(df_clean)*100:.1f}%)")

print("\n5 Beispiele (als Remote erkannt):")
df_clean[df_clean["is_remote_friendly"]][["job_title", "job_city"]].head(5)

## 5️⃣ Rollen-Klassifikation

Wir gruppieren Job-Titel in **Cluster** wie `Data Analyst`, `Data Scientist`, `BI Analyst`. Das macht die spätere Analyse erst möglich — sonst hätten wir 5.000 unterschiedliche Titel-Strings.

**Strategie:** Hierarchische Erkennung von spezifisch (`"Data Analyst"`) zu allgemein (`"daten"` als Catch-All).

In [ ]:
from src.clean_jobs import assign_role_group

df_clean["role_group"] = df_clean.apply(
    lambda r: assign_role_group(r.get("job_title", ""), r.get("job_description", "")),
    axis=1,
)

# Verteilung visualisieren
role_dist = df_clean["role_group"].value_counts()
print("Rollen-Verteilung:")
print(role_dist.to_string())

data_pct = (role_dist[role_dist.index != "Other"].sum() / len(df_clean) * 100)
print(f"\n→ Als Data/Analytics klassifiziert: {data_pct:.1f}%")

## 6️⃣ Vorher-Nachher-Vergleich

Was hat das Cleaning gebracht? Schauen wir uns 3 Beispiel-Jobs vor und nach der Bereinigung an.

In [ ]:
sample_idx = df_clean.head(3).index

for i in sample_idx:
    print("━" * 70)
    print(f"Title:   {df_clean.loc[i, 'job_title'][:65]}")
    print(f"  Stadt:    {df_clean.loc[i, 'job_city']}  (vorher: '{df_raw.loc[i, 'job_city'] if 'job_city' in df_raw.columns else '?'}')")
    print(f"  Junior:   {'✅ ja' if df_clean.loc[i, 'is_junior'] else '— nein'}")
    print(f"  Remote:   {'🏠 ja' if df_clean.loc[i, 'is_remote_friendly'] else '— nein'}")
    print(f"  Rolle:    {df_clean.loc[i, 'role_group']}")
print("━" * 70)

## ✅ Ergebnis

Aus den **rohen API-Daten** haben wir einen **analyse-fähigen Datensatz** gemacht mit folgenden Features:

| Feature | Wert |
|---|---|
| `job_city` | Stadt aus 3 Quellen extrahiert (mit Regex-Fallback) |
| `is_junior` | Bool, basierend auf 8 Keywords |
| `is_remote_friendly` | Bool, basierend auf 14 Regex-Patterns |
| `role_group` | Klassifiziert in 12 Cluster |

**Production-Lauf:** `python -m src.clean_jobs` schreibt nach `data/processed/jobs_cleaned.csv`.

→ Notebook **03** zeigt, wie wir Skills aus den Beschreibungen extrahieren.